# 03 — IOU, Precision/Recall, and mAP From Scratch

This notebook accompanies `03-object-detection-metrics-iou-map-tpr.md`.

It implements, in pure numpy, the exact evaluation mechanics behind the resume's headline result —
**"Attained a TPR of 96% with an IOU threshold set at 0.85"** — on a small set of synthetic predicted
vs. ground-truth boxes:

1. IOU (Intersection over Union) between two boxes.
2. Matching predictions to ground-truth boxes at a chosen IOU threshold to get True
   Positives / False Positives / False Negatives.
3. Precision and TPR (recall) at that threshold.
4. A simplified per-class Average Precision (AP), the building block of mAP.
5. A direct comparison of results at IOU=0.5 (the common default) vs. IOU=0.85 (this project's
   threshold), showing concretely why the stricter threshold is a harder bar to clear.

This is a *toy* dataset built to make the mechanics transparent, not a re-creation of the actual
Indegene evaluation set — the numbers below are illustrative of the method, not the literal 96%
figure (see the note at the end).

## 1. IOU implementation

In [1]:
import numpy as np


def iou(box_a, box_b):
    """Intersection over Union between two (x_min, y_min, x_max, y_max) boxes."""
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b

    inter_x1, inter_y1 = max(xa1, xb1), max(ya1, yb1)
    inter_x2, inter_y2 = min(xa2, xb2), min(ya2, yb2)
    inter_area = max(0.0, inter_x2 - inter_x1) * max(0.0, inter_y2 - inter_y1)

    area_a = (xa2 - xa1) * (ya2 - ya1)
    area_b = (xb2 - xb1) * (yb2 - yb1)
    union_area = area_a + area_b - inter_area

    return inter_area / union_area if union_area > 0 else 0.0


# Sanity checks
assert abs(iou((0, 0, 10, 10), (0, 0, 10, 10)) - 1.0) < 1e-9, "identical boxes -> IOU 1.0"
assert iou((0, 0, 10, 10), (20, 20, 30, 30)) == 0.0, "disjoint boxes -> IOU 0.0"
print("IOU sanity checks passed.")
print("Example: IOU of (0,0,10,10) vs (5,5,15,15) =", round(iou((0, 0, 10, 10), (5, 5, 15, 15)), 4))

IOU sanity checks passed.
Example: IOU of (0,0,10,10) vs (5,5,15,15) = 0.1429


## 2. A small synthetic evaluation set

Five synthetic "images," each with zero, one, or two ground-truth chart boxes. For each image we
also define a model *prediction*: a box plus a confidence score. We deliberately vary how tightly
each prediction matches its ground truth — some are near-perfect, some are loose, and one image has
a spurious false-positive detection with no real chart present at all. This mirrors the kind of
error mix a real detector produces on held-out data.

In [2]:
ground_truths = {
    "img_0": [(50, 50, 250, 200)],
    "img_1": [(30, 40, 220, 260), (300, 60, 500, 220)],
    "img_2": [(80, 100, 380, 320)],
    "img_3": [(60, 60, 260, 240)],
    "img_4": [],  # no chart present in this image
}


def perturb(box, dx1, dy1, dx2, dy2):
    x1, y1, x2, y2 = box
    return (x1 + dx1, y1 + dy1, x2 + dx2, y2 + dy2)


# predictions: (image_id, box, confidence) -- a mix of tight, loose, and spurious detections
predictions = [
    ("img_0", perturb(ground_truths["img_0"][0], 2, -1, -2, 3), 0.98),      # tight -> high IOU
    ("img_1", perturb(ground_truths["img_1"][0], 5, 4, -3, -2), 0.95),      # tight
    ("img_1", perturb(ground_truths["img_1"][1], -20, 15, 25, -18), 0.80),  # loose -> lower IOU
    ("img_2", perturb(ground_truths["img_2"][0], 1, 0, -1, 1), 0.97),       # very tight
    ("img_3", perturb(ground_truths["img_3"][0], -40, -35, 45, 40), 0.55),  # very loose
    ("img_4", (100, 100, 200, 200), 0.60),                                  # false positive
]

for img_id, box, conf in predictions:
    print(f"{img_id}: predicted box {box}, confidence {conf}")

img_0: predicted box (52, 49, 248, 203), confidence 0.98
img_1: predicted box (35, 44, 217, 258), confidence 0.95
img_1: predicted box (280, 75, 525, 202), confidence 0.8
img_2: predicted box (81, 100, 379, 321), confidence 0.97
img_3: predicted box (20, 25, 305, 280), confidence 0.55
img_4: predicted box (100, 100, 200, 200), confidence 0.6


## 3. Matching predictions to ground truth at a chosen IOU threshold

Standard object-detection evaluation: sort predictions by confidence (highest first), then greedily
match each one to the best remaining (unmatched) ground-truth box in the same image. A prediction
counts as a **True Positive** only if its best IOU with an unmatched ground-truth box is **at or
above the threshold** — this is exactly the "IOU threshold defines a correct detection" rule from
Chapter 03. Anything else is a **False Positive**. Ground-truth boxes that never get matched are
**False Negatives**.

In [3]:
def evaluate(predictions, ground_truths, iou_threshold):
    preds_sorted = sorted(predictions, key=lambda p: -p[2])  # highest confidence first
    matched_gt = {img_id: [False] * len(boxes) for img_id, boxes in ground_truths.items()}

    tp = np.zeros(len(preds_sorted))
    fp = np.zeros(len(preds_sorted))

    for i, (img_id, pred_box, conf) in enumerate(preds_sorted):
        gt_boxes = ground_truths.get(img_id, [])
        best_iou, best_j = 0.0, -1
        for j, gt_box in enumerate(gt_boxes):
            if matched_gt[img_id][j]:
                continue  # already matched to a higher-confidence prediction
            cur_iou = iou(pred_box, gt_box)
            if cur_iou > best_iou:
                best_iou, best_j = cur_iou, j

        if best_iou >= iou_threshold and best_j >= 0:
            tp[i] = 1
            matched_gt[img_id][best_j] = True
        else:
            fp[i] = 1

    total_gt = sum(len(b) for b in ground_truths.values())
    tp_count, fp_count = int(tp.sum()), int(fp.sum())
    fn_count = total_gt - tp_count

    precision = tp_count / (tp_count + fp_count) if (tp_count + fp_count) > 0 else 0.0
    recall_tpr = tp_count / total_gt if total_gt > 0 else 0.0  # this IS the "TPR" from the resume bullet

    # Cumulative precision/recall curve (standard AP computation ingredients)
    cum_tp, cum_fp = np.cumsum(tp), np.cumsum(fp)
    precisions = cum_tp / (cum_tp + cum_fp)
    recalls = cum_tp / total_gt if total_gt > 0 else np.zeros_like(cum_tp)

    # Simplified AP: integrate precision over recall via the trapezoidal rule
    # (a simplified stand-in for COCO/VOC's more elaborate interpolated-AP recipes)
    order = np.argsort(recalls)
    r_sorted, p_sorted = recalls[order], precisions[order]
    if len(r_sorted) > 1:
        ap = float(np.sum(np.diff(r_sorted) * (p_sorted[1:] + p_sorted[:-1]) / 2.0))
    else:
        ap = 0.0

    return {
        "TP": tp_count, "FP": fp_count, "FN": fn_count,
        "precision": round(precision, 3),
        "TPR_recall": round(recall_tpr, 3),
        "simplified_AP": round(ap, 3),
    }


result_at_085 = evaluate(predictions, ground_truths, iou_threshold=0.85)
print("Evaluation at IOU threshold = 0.85 (this project's threshold):")
print(result_at_085)

Evaluation at IOU threshold = 0.85 (this project's threshold):
{'TP': 3, 'FP': 3, 'FN': 2, 'precision': 0.5, 'TPR_recall': 0.6, 'simplified_AP': 0.4}


## 4. Comparing IOU=0.5 (common default) vs. IOU=0.85 (this project)

This is the core "why 0.85 is a harder bar than 0.5" demonstration from Chapter 03: the same set of
predictions, evaluated under two different definitions of "correct." Watch TPR (recall) drop as the
matching requirement gets stricter — some predictions that were "close enough" at 0.5 no longer
qualify at 0.85.

In [4]:
for threshold in [0.5, 0.85]:
    result = evaluate(predictions, ground_truths, iou_threshold=threshold)
    print(f"IOU threshold = {threshold}:  {result}")

result_50 = evaluate(predictions, ground_truths, iou_threshold=0.5)
result_85 = evaluate(predictions, ground_truths, iou_threshold=0.85)

assert result_85["TPR_recall"] <= result_50["TPR_recall"], \
    "a stricter IOU threshold should never increase recall"
print("\nConfirmed: stricter IOU threshold (0.85) yields lower-or-equal TPR than the lenient")
print("default (0.5) -- exactly the effect Chapter 03 describes. A model reporting a HIGH TPR")
print("at the stricter 0.85 threshold (like the real project's 96%) is therefore a stronger claim")
print("than reporting the same TPR number at 0.5 would be.")

IOU threshold = 0.5:  {'TP': 4, 'FP': 2, 'FN': 1, 'precision': 0.667, 'TPR_recall': 0.8, 'simplified_AP': 0.6}
IOU threshold = 0.85:  {'TP': 3, 'FP': 3, 'FN': 2, 'precision': 0.5, 'TPR_recall': 0.6, 'simplified_AP': 0.4}

Confirmed: stricter IOU threshold (0.85) yields lower-or-equal TPR than the lenient
default (0.5) -- exactly the effect Chapter 03 describes. A model reporting a HIGH TPR
at the stricter 0.85 threshold (like the real project's 96%) is therefore a stronger claim
than reporting the same TPR number at 0.5 would be.


## Note on the numbers in this notebook

This toy dataset has only 5 predictions and 5 ground-truth boxes, deliberately chosen to make the
matching/threshold mechanics easy to trace by eye -- it is **not** a reproduction of the real
Indegene evaluation set, and the resulting TPR values here (well below 96%) shouldn't be read as
contradicting the resume's real result. The point of this notebook is the *mechanism*: exactly how a
"TPR at IOU threshold X" number is computed, and why raising the IOU threshold from 0.5 to 0.85 makes
that number harder to earn. Scale this same `evaluate()` function up to a real validation set of
hundreds of images and it computes precisely the metric the resume bullet quotes.